# Notebook 06 — Optimisation des Paramètres (version Kaggle)

**Dataset :** `chatoditlokonga/dataset-data-science-92iemj`  
**Accélérateur recommandé :** GPU (T4 ou P100)

Version simplifiée : **embeddings uniquement** (rank_bm25 retiré).

Résultats connus du run local :
- Meilleure stratégie : `text` seul → MRR@10 = 0.4635 (+68.9% vs baseline)
- Score Kaggle v7 : **0.28328**

In [ ]:
import os

BASE        = '/kaggle/input/datasets/chatoditlokonga/dataset-data-science-92iemj'
DATA_DIR    = f'{BASE}/data'
MODELS_DIR  = f'{BASE}/models'
OUTPUTS_DIR = '/kaggle/working'

print('Chemins configurés :')
print(f'  DATA    : {DATA_DIR}')
print(f'  MODELS  : {MODELS_DIR}')
print(f'  OUTPUTS : {OUTPUTS_DIR}')
print()
print('Fichiers data   :', os.listdir(DATA_DIR))
print('Fichiers models :', os.listdir(MODELS_DIR))

## Cellule 1 — Imports et Chargement

In [ ]:
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# --- Corpus ---
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

# --- Embeddings pré-calculés ---
corpus_embeddings = np.load(os.path.join(MODELS_DIR, 'corpus_embeddings.npy'))

# --- Sentence Transformer ---
model_st = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Corpus         : {len(df_docs):,} documents')
print(f'Embeddings     : {corpus_embeddings.shape}')
print(f'Queries train  : {len(queries_train)}')
print(f'Queries test   : {len(queries_test)}')
print('Chargement OK.')

## Cellule 2 — Fonctions d'Évaluation

`evaluate_emb_batch` encode toutes les requêtes en une seule passe GPU — ~10× plus rapide que le mode séquentiel.

In [ ]:
def get_relevant_ids(qgts, query_id):
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]


def _recall_at_k(retrieved, relevant_ids, k):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    return len(set(retrieved[:k]) & rel_set) / max(len(rel_set), 1)


def _precision_at_k(retrieved, relevant_ids, k):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    return len(set(retrieved[:k]) & rel_set) / k


def _mrr(retrieved, relevant_ids):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    for rank, idx in enumerate(retrieved, start=1):
        if idx in rel_set:
            return 1.0 / rank
    return 0.0


def evaluate_emb_batch(queries, qgts, k=10, batch_size=128):
    """
    Encode toutes les requêtes en une seule passe GPU.
    ~10× plus rapide que les appels séquentiels.
    """
    valid = [(q, get_relevant_ids(qgts, q['id'])) for q in queries]
    valid = [(q, rel) for q, rel in valid if rel]
    texts = [q['_query_text'] for q, _ in valid]

    q_embs        = model_st.encode(texts, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=False)
    scores_matrix = cosine_similarity(q_embs, corpus_embeddings)

    mrrs, recs, precs = [], [], []
    for (_, rel), scores in zip(valid, scores_matrix):
        top_k = np.argsort(scores)[::-1][:k].tolist()
        mrrs.append(_mrr(top_k, rel))
        recs.append(_recall_at_k(top_k, rel, k))
        precs.append(_precision_at_k(top_k, rel, k))

    return {
        f'MRR@{k}':       round(float(np.mean(mrrs)),  4),
        f'Recall@{k}':    round(float(np.mean(recs)),  4),
        f'Precision@{k}': round(float(np.mean(precs)), 4),
        'n_queries':      len(mrrs),
    }


print('Fonctions d\'évaluation prêtes.')

## Cellule 3 — Tuning : Stratégie de Construction de Requête

5 variantes testées sur embeddings avec encodage batch GPU.

In [ ]:
QUERY_STRATEGIES = {
    'text':            lambda q: q.get('text', '').strip(),
    'text+title':      lambda q: ' '.join(p for p in [q.get('text', ''), q.get('title', '')] if p).strip(),
    'text+title+tags': lambda q: ' '.join(p for p in [
                                      q.get('text', ''), q.get('title', ''),
                                      ' '.join(q['tags']) if q.get('tags') else ''
                                  ] if p).strip(),
    'title+tags':      lambda q: ' '.join(p for p in [
                                      q.get('title', ''),
                                      ' '.join(q['tags']) if q.get('tags') else ''
                                  ] if p).strip(),
    'tags':            lambda q: ' '.join(q['tags']) if q.get('tags') else q.get('text', '').strip(),
}

K_EVAL       = 10
rows_strategy = []

for strat_name, build_fn in QUERY_STRATEGIES.items():
    queries_annotated = [dict(q, _query_text=build_fn(q)) for q in queries_train]
    metrics = evaluate_emb_batch(queries_annotated, qgts_train, k=K_EVAL)
    rows_strategy.append({'Stratégie': strat_name, **metrics})
    print(f'  [{strat_name:20s}]  MRR@{K_EVAL}={metrics[f"MRR@{K_EVAL}"]:.4f}  '
          f'Recall@{K_EVAL}={metrics[f"Recall@{K_EVAL}"]:.4f}')

df_strategy = pd.DataFrame(rows_strategy).set_index('Stratégie')

print()
print('=== Tableau : Impact de la stratégie de requête ===')
display(df_strategy.style
        .format('{:.4f}', subset=[c for c in df_strategy.columns if c != 'n_queries'])
        .highlight_max(axis=0, subset=[f'MRR@{K_EVAL}'], color='lightgreen')
        .highlight_min(axis=0, subset=[f'MRR@{K_EVAL}'], color='#ffcccc'))

best_strategy_name = df_strategy[f'MRR@{K_EVAL}'].idxmax()
best_strategy_fn   = QUERY_STRATEGIES[best_strategy_name]
best_strategy_mrr  = df_strategy[f'MRR@{K_EVAL}'].max()
baseline_mrr       = df_strategy.loc['text+title+tags', f'MRR@{K_EVAL}']

print(f'\nMeilleure stratégie : "{best_strategy_name}"  MRR@{K_EVAL}={best_strategy_mrr:.4f}')
print(f'Baseline (text+title+tags)       : MRR@{K_EVAL}={baseline_mrr:.4f}')
print(f'Gain                             : {best_strategy_mrr - baseline_mrr:+.4f} ({(best_strategy_mrr/baseline_mrr - 1)*100:+.1f}%)')

## Cellule 4 — Résumé : Tableau Comparatif Final

In [ ]:
# Baseline (stratégie par défaut text+title+tags)
queries_default    = [dict(q, _query_text=QUERY_STRATEGIES['text+title+tags'](q)) for q in queries_train]
queries_best_strat = [dict(q, _query_text=best_strategy_fn(q))                    for q in queries_train]

baseline_emb = evaluate_emb_batch(queries_default,    qgts_train, k=K_EVAL)
tuned_emb    = evaluate_emb_batch(queries_best_strat, qgts_train, k=K_EVAL)

summary_rows = [
    {'Config': '[BASELINE] Emb — text+title+tags',          **{k: v for k, v in baseline_emb.items() if k != 'n_queries'}},
    {'Config': f'[TUNED]    Emb — {best_strategy_name}',    **{k: v for k, v in tuned_emb.items()    if k != 'n_queries'}},
]

df_summary = pd.DataFrame(summary_rows).set_index('Config')

print('=== TABLEAU COMPARATIF FINAL ===')
display(df_summary.style
        .format('{:.4f}')
        .highlight_max(axis=0, color='lightgreen')
        .highlight_min(axis=0, color='#ffcccc')
        .set_caption('Baseline vs Tuned — Embeddings'))

winner_mrr   = tuned_emb[f'MRR@{K_EVAL}']
baseline_mrr = baseline_emb[f'MRR@{K_EVAL}']
gain         = winner_mrr - baseline_mrr

print()
print(f'Gagnant      : Emb — {best_strategy_name}')
print(f'MRR@{K_EVAL}     : {winner_mrr:.4f}')
print(f'Gain         : {gain:+.4f} ({gain/baseline_mrr*100:+.1f}%)')
print(f'\nConfiguration retenue : Embeddings + stratégie="{best_strategy_name}"')

## Cellule 5 — Génération de la Soumission (`92iemj_v7_tuned.csv`)

In [ ]:
K_SUBMIT = 100

# Encodage batch de toutes les requêtes test en une seule passe GPU
test_texts = [best_strategy_fn(q) for q in queries_test]
q_embs     = model_st.encode(test_texts, batch_size=128, convert_to_numpy=True, show_progress_bar=True)

# Similarité cosinus en une opération matricielle
scores_mat  = cosine_similarity(q_embs, corpus_embeddings)
top_indices = [np.argsort(scores)[::-1][:K_SUBMIT].tolist() for scores in scores_mat]

rows_submit = []
for q_entry, top_k in zip(queries_test, top_indices):
    rows_submit.append({
        'query_id':         q_entry['id'],
        'relevant_doc_ids': json.dumps([idx_to_id[i] for i in top_k]),
        'category':         q_entry.get('category', '?') or '?',
    })

submission_path = os.path.join(OUTPUTS_DIR, '92iemj_v7_tuned.csv')
df_submit = pd.DataFrame(rows_submit)
df_submit.to_csv(submission_path, index=False)

# Vérifications
assert len(df_submit) == 141, f'ERREUR : attendu 141 lignes, obtenu {len(df_submit)}'
assert (df_submit['relevant_doc_ids'].apply(lambda x: len(json.loads(x))) == K_SUBMIT).all()

print(f'Soumission sauvegardée : {submission_path}')
print(f'Lignes : {len(df_submit)} requêtes × top-{K_SUBMIT} documents')
print('Vérifications OK.')
print()
print('=== Configuration finale ===')
print(f'  Modèle    : Embeddings all-MiniLM-L6-v2')
print(f'  Stratégie : {best_strategy_name}')
print(f'  MRR@10    : {winner_mrr:.4f}  (baseline : {baseline_mrr:.4f})')
print(f'  Fichier   : {submission_path}')
print()
display(df_submit.head(3))